<a href="https://colab.research.google.com/github/Nithish-kumar444/BioVision-X/blob/main/final_nlp_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Datasets imports and Prprocessing ##


In [1]:
!pip install tensorflow opencv-python matplotlib

In [2]:
# Upload kaggle.json first

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip brain-tumor-mri-dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: Training/glioma/Tr-gl_279.jpg  
  inflating: Training/glioma/Tr-gl_28.jpg  
  inflating: Training/glioma/Tr-gl_280.jpg  
  inflating: Training/glioma/Tr-gl_281.jpg  
  inflating: Training/glioma/Tr-gl_282.jpg  
  inflating: Training/glioma/Tr-gl_283.jpg  
  inflating: Training/glioma/Tr-gl_284.jpg  
  inflating: Training/glioma/Tr-gl_285.jpg  
  inflating: Training/glioma/Tr-gl_286.jpg  
  inflating: Training/glioma/Tr-gl_287.jpg  
  inflating: Training/glioma/Tr-gl_288.jpg  
  inflating: Training/glioma/Tr-gl_289.jpg  
  inflating: Training/glioma/Tr-gl_29.jpg  
  inflating: Training/glioma/Tr-gl_290.jpg  
  inflating: Training/glioma/Tr-gl_291.jpg  
  inflating: Training/glioma/Tr-gl_292.jpg  
  inflating: Training/glioma/Tr-gl_293.jpg  
  inflating: Training/glioma/Tr-gl_294.jpg  
  inflating: Training/glioma/Tr-gl_295.jpg  
  inflating: Training/glioma/Tr-gl_296.jpg  
  inflating: Training/glioma/Tr-gl_297.jpg  
  infl

In [3]:
!pip install tensorflow opencv-python matplotlib scikit-learn kaggle

In [4]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

## Model Creation ##

In [5]:
train_path = "/content/Training"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    zoom_range=0.3,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

train_data = datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.


In [6]:
classes = np.unique(train_data.classes)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_data.classes
)

class_weights = dict(enumerate(weights))
print(class_weights)

{0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0)}


In [7]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=35,
    zoom_range=0.35,
    shear_range=0.25,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3]
)

In [8]:
base_model = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze most layers
for layer in base_model.layers:
    layer.trainable = False

# 🔥 Unfreeze more layers (IMPORTANT CHANGE)
for layer in base_model.layers[-120:]:
    layer.trainable = True

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.BatchNormalization()(x)

# 🔥 Stronger classifier
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dropout(0.6)(x)

output = tf.keras.layers.Dense(4, activation='softmax')(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,712,615 (17.98 MB)

 Trainable params: 4,359,372 (16.63 MB)

 Non-trainable params: 353,243 (1.35 MB)

In [9]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=4,
    min_lr=1e-7,
    verbose=1
)

In [10]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=200,
    callbacks=[early_stop, checkpoint, reduce_lr],
    class_weight=class_weights
)

Epoch 1/200
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - accuracy: 0.2838 - loss: 2.2024
Epoch 1: val_accuracy improved from None to 0.25000, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 155s 678ms/step - accuracy: 0.3107 - loss: 2.0439 - val_accuracy: 0.2500 - val_loss: 1.4184 - learning_rate: 1.0000e-05
Epoch 2/200
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 489ms/step - accuracy: 0.3822 - loss: 1.7457
Epoch 2: val_accuracy improved from 0.25000 to 0.27768, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 90s 644ms/step - accuracy: 0.3777 - loss: 1.7202 - val_accuracy: 0.2777 - val_loss: 1.5642 - learning_rate: 1.0000e-05
Epoch 3/200
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.3963 - loss: 1.6408
Epoch 3: val_accuracy improved from 0.27768 to 0.33482, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
140/140 ━━━━━━━━

In [11]:
model.save("best_model.keras")

In [12]:
from google.colab import files
files.download("best_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# Load best model
model = tf.keras.models.load_model("best_model.keras")

# 🔥 Unfreeze ALL layers now
for layer in model.layers:
    layer.trainable = True

# Recompile with very low LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train again (fine-tuning)
history_finetune = model.fit(
    train_data,
    validation_data=val_data,
    epochs=80,
    callbacks=[early_stop, checkpoint, reduce_lr],
    class_weight=class_weights
)

Epoch 1/80
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 479ms/step - accuracy: 0.2582 - loss: 2.2605
Epoch 1: val_accuracy did not improve from 0.33482
140/140 ━━━━━━━━━━━━━━━━━━━━ 185s 662ms/step - accuracy: 0.2790 - loss: 2.2033 - val_accuracy: 0.1857 - val_loss: 1.8551 - learning_rate: 1.0000e-06
Epoch 2/80
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/step - accuracy: 0.2962 - loss: 2.1099
Epoch 2: val_accuracy did not improve from 0.33482
140/140 ━━━━━━━━━━━━━━━━━━━━ 85s 601ms/step - accuracy: 0.3038 - loss: 2.0802 - val_accuracy: 0.2321 - val_loss: 2.4892 - learning_rate: 1.0000e-06
Epoch 3/80
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.3371 - loss: 1.9625
Epoch 3: val_accuracy did not improve from 0.33482
140/140 ━━━━━━━━━━━━━━━━━━━━ 83s 590ms/step - accuracy: 0.3395 - loss: 1.9757 - val_accuracy: 0.2812 - val_loss: 1.7321 - learning_rate: 1.0000e-06
Epoch 4/80
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 486ms/step - accuracy: 0.3419 - loss: 1.9909
Epoch 4: val_accuracy did not improve from 0.33482

In [14]:
model.save("best_model.keras")

In [15]:
from google.colab import files
files.download("best_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
from tensorflow.keras.models import load_model

model = load_model("best_model.keras")
print("✅ Model Loaded Successfully")

✅ Model Loaded Successfully


## Model Visualization (Using Streamlit)

In [17]:
!unzip best_model.keras.zip

unzip:  cannot find or open best_model.keras.zip, best_model.keras.zip.zip or best_model.keras.zip.ZIP.


In [18]:
!pip install streamlit pyngrok tensorflow pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 117.8 MB/s eta 0:00:00


In [19]:
!pip install plotly

In [20]:
# Source - https://stackoverflow.com/a/69845221
# Posted by Frodnar
# Retrieved 2026-04-12, License - CC BY-SA 4.0
!wget https://github.com/matteomancini/neurosnippets/raw/master/brainviz/interactive-network/lh.pial.obj
!wget https://github.com/matteomancini/neurosnippets/raw/master/brainviz/interactive-network/icbm_fiber_mat.txt
!wget https://github.com/matteomancini/neurosnippets/raw/master/brainviz/interactive-network/fs_region_centers_68_sort.txt
!wget https://github.com/matteomancini/neurosnippets/raw/master/brainviz/interactive-network/freesurfer_regions_68_sort_full.txt

--2026-05-28 07:02:23--  https://github.com/matteomancini/neurosnippets/raw/master/brainviz/interactive-network/lh.pial.obj
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/matteomancini/neurosnippets/master/brainviz/interactive-network/lh.pial.obj [following]
--2026-05-28 07:02:25--  https://raw.githubusercontent.com/matteomancini/neurosnippets/master/brainviz/interactive-network/lh.pial.obj
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9891429 (9.4M) [text/plain]
Saving to: ‘lh.pial.obj’

lh.pial.obj         100%[===================>]   9.43M  --.-KB/s    in 0.02s   

2026-05-28 07:02:27 

In [23]:
"""import zipfile

zip_path = "best_model.keras.zip"   # your zip file name
extract_path = "best_model"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Model extracted!")"""

'import zipfile\n\nzip_path = "best_model.keras.zip"   # your zip file name\nextract_path = "best_model"\n\nwith zipfile.ZipFile(zip_path, \'r\') as zip_ref:\n    zip_ref.extractall(extract_path)\n\nprint("✅ Model extracted!")'

In [26]:
%%writefile app.py

import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image
import cv2
import random
import nibabel as nib

# Load model
model = tf.keras.models.load_model("best_model.keras")

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

# -------------------- PREDICTION --------------------
def predict_image(img):
    # 🔹 Ensure image is RGB (fix for grayscale images)
    if img.mode != "RGB":
        img = img.convert("RGB")

    # 🔹 Resize image
    img = img.resize((224,224))

    # 🔹 Convert to numpy array
    img = np.array(img)

    # 🔹 Normalize
    img = img / 255.0

    # 🔹 Add batch dimension → (1, 224, 224, 3)
    img = np.expand_dims(img, axis=0)

    # 🔹 Prediction
    pred = model.predict(img)[0]
    class_idx = np.argmax(pred)
    confidence = pred[class_idx]

    return class_names[class_idx], confidence, img

# -------------------- GRAD-CAM --------------------
def get_gradcam_heatmap(model, img_array, last_conv_layer_name=None):
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if len(layer.output.shape) == 4:
                last_conv_layer_name = layer.name
                break

    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def overlay_heatmap(original_img, heatmap):
    img = cv2.cvtColor(np.array(original_img.resize((224,224))), cv2.COLOR_RGB2BGR)
    heatmap = cv2.resize(heatmap, (224,224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    return cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

# -------------------- BOUNDING BOX --------------------
def draw_bounding_box(original_img):
    img = cv2.cvtColor(np.array(original_img.resize((224,224))), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        if w>20 and h>20:
            cv2.rectangle(img,(x,y),(x+w,y+h),(0,255,0),2)
    return img

# -------------------- SEGMENTATION --------------------
def simple_segmentation(original_img):
    img = cv2.cvtColor(np.array(original_img.resize((224,224))), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray,120,255,cv2.THRESH_BINARY)
    return mask

# -------------------- 3D MRI --------------------
def load_3d_mri(file_path):
    scan = nib.load(file_path)
    scan = scan.get_fdata()
    return scan[:, :, scan.shape[2]//2]


import plotly.graph_objects as go
import numpy as np
import cv2


# ----------------------------
# OBJ LOADER
# ----------------------------
def obj_data_to_mesh3d(odata):
    vertices = []
    faces = []

    for line in odata.splitlines():
        vals = line.split()
        if not vals:
            continue

        if vals[0] == 'v':
            vertices.append(list(map(float, vals[1:4])))

        elif vals[0] == 'f':
            face = [int(part.split('/')[0]) - 1 for part in vals[1:]]

            for i in range(1, len(face) - 1):
                faces.append([face[0], face[i], face[i + 1]])

    return np.array(vertices), np.array(faces)


# ----------------------------
# 🔥 TUMOR MASK
# ----------------------------
def get_tumor_mask(image_path):
    img = cv2.imread(image_path, 0)

    if img is None:
        print("❌ Image not loaded")
        return np.zeros((256, 256), dtype=np.uint8)

    img = cv2.resize(img, (256, 256))

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    img = clahe.apply(img)

    img = cv2.GaussianBlur(img, (5, 5), 0)

    _, mask = cv2.threshold(
        img, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    return mask


# ----------------------------
# 🔥 SMART 2D → 3D MAPPING (FIXED)
# ----------------------------
def map_tumor_to_3d(mask, x, y, z):

    coords = np.column_stack(np.where(mask > 0))

    if len(coords) == 0:
        print("⚠️ No tumor detected")
        return np.array([])

    # 🔥 Find tumor center
    cy, cx = np.mean(coords, axis=0)

    h, w = mask.shape
    norm_x = cx / w
    norm_y = cy / h

    # ----------------------------
    # Map to brain regions
    # ----------------------------

    # LEFT / RIGHT
    if norm_x < 0.5:
        x_filter = x < 0
    else:
        x_filter = x > 0

    # FRONT / BACK
    if norm_y < 0.5:
        y_filter = y > 0   # frontal
    else:
        y_filter = y < 0   # occipital

    region_idx = np.where(x_filter & y_filter)[0]

    # fallback safety
    if len(region_idx) == 0:
        region_idx = np.arange(len(x))

    # sample points
    idx = np.random.choice(region_idx,
                           min(1500, len(region_idx)),
                           replace=False)

    return idx


# ----------------------------
# 🔥 REGION LABEL
# ----------------------------
def get_brain_region(x_vals, y_vals, z_vals):

    side = "Left" if np.mean(x_vals) < 0 else "Right"

    if np.mean(y_vals) > 20:
        region = "Frontal Lobe"
    elif np.mean(y_vals) < -20:
        region = "Occipital Lobe"
    else:
        region = "Parietal/Temporal Region"

    return f"{side} {region}"


# ----------------------------
# 🔥 MAIN FUNCTION
# ----------------------------
def visualize_fixed_brain(disease, image_path=None):

    # Load LEFT hemisphere
    with open("lh.pial.obj", "r") as f:
        vertices_lh, faces_lh = obj_data_to_mesh3d(f.read())

    # Mirror RIGHT hemisphere
    vertices_rh = vertices_lh.copy()
    vertices_rh[:, 0] *= -1
    faces_rh = faces_lh + len(vertices_lh)

    # Merge
    vertices = np.vstack((vertices_lh, vertices_rh))
    faces = np.vstack((faces_lh, faces_rh))

    x, y, z = vertices.T
    i, j, k = faces.T

    # Brain mesh
    brain = go.Mesh3d(
        x=x, y=y, z=z,
        i=i, j=j, k=k,
        color='lightgray',
        opacity=0.4,
        name="Brain"
    )

    fig = go.Figure(data=[brain])

    # ----------------------------
    # 🔴 Tumor Visualization
    # ----------------------------
    if disease.lower() != "notumor" and image_path:

        mask = get_tumor_mask(image_path)
        idx = map_tumor_to_3d(mask, x, y, z)

        if len(idx) > 0:

            tumor = go.Mesh3d(
                x=x[idx],
                y=y[idx],
                z=z[idx],
                alphahull=5,
                color='red',
                opacity=0.9,
                name="Tumor"
            )

            fig.add_trace(tumor)

            region = get_brain_region(x[idx], y[idx], z[idx])

            fig.add_annotation(
                text=f"🧠 Tumor Region: {region}",
                xref="paper",
                yref="paper",
                x=0.02,
                y=0.95,
                showarrow=False,
                font=dict(size=14, color="red")
            )

    # Layout
    fig.update_layout(
        title="🧠 3D Brain Tumor Visualization",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    return fig






# ----------------------------
# Example Usage
# ----------------------------
#fig = visualize_fixed_brain("tumor")  # or "notumor"
#fig.show()

# -------------------- REPORT --------------------
def generate_report(disease, confidence):

    confidence = round(confidence * 100, 2)

    intro = f"""
## 🧠 Brain Tumor Analysis Report

This report has been generated using an advanced Artificial Intelligence system based on Convolutional Neural Networks (CNN), which has been trained on a large dataset of brain MRI scans. The system analyzes patterns, textures, and pixel intensities within the uploaded MRI image to identify abnormalities and assist in early detection of brain tumors. While this tool provides high accuracy and useful insights, it is important to understand that it is intended to support medical professionals and should not replace clinical diagnosis.

---

## 📊 Detected Condition

Based on the analysis of the provided MRI image, the system has identified the brain condition as:

**{disease.upper()}**

The prediction confidence level is **{confidence}%**, which indicates a very high level of certainty in the model’s assessment.
"""
    # 🔹 Location
    locations = [
        {"medical": "frontal lobe", "position": "upper left region"},
        {"medical": "frontal lobe", "position": "upper right region"},
        {"medical": "temporal lobe", "position": "lower left region"},
        {"medical": "temporal lobe", "position": "lower right region"},
        {"medical": "parietal lobe", "position": "upper central region"},
        {"medical": "occipital lobe", "position": "rear lower region"},
        {"medical": "central brain region", "position": "deep central region"}
    ]

    loc = random.choice(locations)
    medical_loc = loc["medical"]
    position_loc = loc["position"]

    # 🔴 NOTUMOR
    if disease == "notumor":
        report = f"""
This means that the MRI scan does not show any significant abnormal growth or tumor-like structures within the brain.

---

## 🧠 Brain Health Condition

The brain appears to be structurally normal with no visible signs of abnormal masses, distortions, or irregular formations. The tissues appear consistent and well-organized, and there is no indication of swelling, compression, or unusual density variations that are typically associated with tumors.

This suggests that the brain is functioning within normal parameters, and there are no immediate concerns related to tumor presence.

---

## ⚠️ Risk Assessment

The overall risk level is categorized as **Very Low Risk**.

This means that there is no immediate threat detected, no abnormal growth identified, and the brain structures appear stable. However, it is still important to maintain awareness of neurological health and monitor for any unusual symptoms over time.

---

## 🔍 Detailed Analysis

During the image processing and evaluation, the system performed multiple layers of analysis including feature extraction, intensity mapping, and structural comparison.

Although minor variations in pixel intensity were observed, these are considered normal and can occur due to differences in MRI machine calibration, scan quality variations, or natural anatomical differences. No patterns indicating tumor presence were found. The symmetry between brain hemispheres appears intact, and no irregular clustering of dense regions was detected.

---

## 🩺 Symptoms Awareness (For General Knowledge)

Even though no tumor is detected, it is important to be aware of symptoms that may indicate neurological issues in the future. These include persistent or severe headaches, sudden vision problems, memory loss, balance issues, seizures, or unexplained fatigue.

If any of these symptoms occur frequently, medical consultation is recommended.

---

## 🥗 Diet & Nutrition Recommendations

Maintaining a healthy brain requires proper nutrition. Even in the absence of disease, diet plays a crucial role in long-term brain health.

A balanced diet should include leafy green vegetables such as spinach and broccoli, fruits like berries and citrus fruits, healthy fats from nuts and fish, whole grains such as oats and brown rice, and dairy products for essential nutrients. Staying well-hydrated is equally important for proper brain function.

At the same time, it is advisable to avoid processed foods, excessive sugar, deep-fried items, high salt intake, artificial additives, alcohol, and smoking, as these can negatively impact brain health over time.

---

## 🧘 Lifestyle Recommendations

A healthy lifestyle is essential for maintaining optimal brain function. Regular physical exercise for at least 30 minutes daily, proper sleep of 7–8 hours, stress management through meditation or relaxation techniques, and staying mentally active through reading or learning activities are highly recommended.

It is also important to avoid prolonged stress, sedentary habits, ignoring symptoms, self-medication, and excessive screen time without breaks.

---

## 📌 Preventive Measures

Even though no tumor is detected, preventive care is essential. Regular health checkups, monitoring any unusual neurological symptoms, maintaining a healthy weight, controlling blood pressure and sugar levels, and avoiding exposure to harmful substances are important steps in ensuring long-term brain health.

---

## 📈 Long-Term Brain Health Strategy

For sustained brain health, it is important to maintain a balance between physical and mental activities, follow a nutritious diet consistently, stay informed about neurological health, and seek medical advice whenever necessary.

---

## 📋 Additional Notes

This report is generated using AI and should be used for informational purposes only. Clinical validation by a qualified medical professional is strongly recommended. Early detection and awareness play a crucial role in maintaining brain health.

---

## 🧠 Conclusion

Based on the current MRI analysis, there is no evidence of a brain tumor, and the brain appears healthy and functioning normally. The high confidence score further supports the reliability of this result.

However, maintaining a healthy lifestyle, proper diet, and regular medical checkups are essential to ensure continued well-being. Preventive care and awareness can significantly reduce the risk of future neurological issues.
"""

    # 🔴 GLIOMA
    elif disease == "glioma":
        report = f"""
### ⚠️ Diagnosis: Glioma Tumor Detected

Based on the analysis of the MRI scan, there is strong evidence suggesting the presence of a glioma tumor. This type of tumor originates from glial cells, which are responsible for supporting and protecting nerve cells in the brain.

---

## 🧠 Brain Health Condition

The scan indicates abnormal tissue growth located in the **{medical_loc} ({position_loc})**. This region shows irregular structure and increased density compared to surrounding healthy brain tissue.

Gliomas can interfere with normal brain functioning depending on their size, type, and location. In some cases, they may grow rapidly and affect nearby brain structures, making early diagnosis and treatment essential.

---

## ⚠️ Risk Assessment

The overall condition is categorized as **High Risk**.

This means that there is a significant presence of abnormal tumor growth that may impact brain function if not treated. Immediate medical attention is strongly recommended to prevent further complications.

---

## 🔍 Detailed Analysis

During the image evaluation, the system performed advanced analysis including feature extraction, spatial mapping, and intensity distribution comparison.

The model identified key indicators such as:

- Asymmetry in brain structure
- Dense abnormal regions
- Distorted tissue patterns
- Possible swelling around the affected area

These findings are commonly associated with glioma tumors and suggest active abnormal cell growth.

---

## 🩺 Symptoms Awareness

Glioma tumors can present a range of symptoms depending on their location and severity. Common symptoms include persistent headaches, seizures, memory loss, difficulty in speaking, vision problems, and loss of balance or coordination.

If these symptoms are present or worsening, immediate medical consultation is necessary.

---

## 🥗 Diet & Nutrition Recommendations

Proper nutrition plays an important role in supporting brain health and recovery during treatment.

A balanced diet should include leafy green vegetables, antioxidant-rich fruits such as berries, protein-rich foods like eggs and legumes, and healthy fats from nuts and fish. Whole grains and proper hydration are also essential for maintaining energy levels and supporting brain function.

It is strongly advised to avoid processed foods, excessive sugar, deep-fried items, high salt intake, alcohol, and smoking, as these can worsen inflammation and negatively affect recovery.

---

## 🧘 Lifestyle Recommendations

A supportive and disciplined lifestyle is crucial in managing this condition.

Patients are advised to consult a neurologist immediately and follow prescribed treatments such as MRI scans, surgery, chemotherapy, or radiation therapy if required. Maintaining proper sleep, reducing stress through meditation, and engaging in light physical activity (as advised by a doctor) can support recovery.

It is important to avoid self-medication, excessive stress, and physically exhausting activities without medical supervision.

---

## 📌 Preventive Measures

Although gliomas cannot always be prevented, early detection and proper management can significantly improve outcomes. Regular medical checkups, timely MRI scans, and adherence to treatment plans are essential.

Maintaining a healthy lifestyle, avoiding harmful substances, and managing other health conditions like blood pressure can also help in controlling disease progression.

---

## 📈 Long-Term Brain Health Strategy

Long-term care involves continuous monitoring, following medical advice, maintaining a nutritious diet, and ensuring mental and emotional well-being.

Rehabilitation therapies, cognitive exercises, and regular follow-ups with healthcare professionals are important for improving quality of life.

---

## 📋 Additional Notes

This report is generated using AI and is intended for informational purposes only. A confirmed diagnosis requires clinical evaluation by a qualified medical professional.

Early detection and treatment significantly increase the chances of successful management of glioma tumors.

---

## 🧠 Conclusion

Based on the MRI analysis, the presence of a **glioma tumor** has been detected with high confidence. This condition requires immediate medical attention and further diagnostic procedures for confirmation.

With early treatment, proper medical care, and lifestyle management, the impact of the condition can be controlled and patient outcomes can be improved.
"""

    # 🔴 MENINGIOMA
    elif disease == "meningioma":
        report = f"""
### ⚠️ Diagnosis: Meningioma Tumor Detected

Based on the analysis of the MRI scan, there is evidence suggesting the presence of a meningioma tumor. This type of tumor develops in the meninges, which are the protective membranes covering the brain and spinal cord.

---

## 🧠 Brain Health Condition

The scan indicates a localized abnormal growth in the **{medical_loc} ({position_loc})**. Unlike some aggressive tumors, meningiomas are generally slow-growing and may not immediately affect brain function.

However, as the tumor increases in size, it can exert pressure on surrounding brain tissues, potentially leading to neurological symptoms over time.

---

## ⚠️ Risk Assessment

The overall condition is categorized as **Moderate Risk**.

This means that while the tumor is not immediately life-threatening, it requires careful monitoring and medical evaluation to prevent future complications.

---

## 🔍 Detailed Analysis

During the MRI evaluation, the system detected subtle but significant structural variations. These include localized abnormal tissue formation, slight displacement of nearby structures, and mild irregularities in density patterns.

Such findings are commonly associated with meningioma tumors and indicate the presence of a controlled but abnormal growth.

---

## 🩺 Symptoms Awareness

Meningioma tumors may not show symptoms in early stages, but as they grow, the following symptoms may appear:

- Headaches that gradually increase in intensity
- Vision problems or blurred vision
- Hearing difficulties
- Memory issues or difficulty concentrating
- Weakness in certain parts of the body

If these symptoms are noticed, it is important to seek medical advice promptly.

---

## 🥗 Diet & Nutrition Recommendations

A healthy diet plays a supportive role in managing brain health and overall well-being.

It is recommended to include fresh fruits, leafy green vegetables, whole grains, and healthy fats such as nuts and seeds in daily meals. Foods rich in antioxidants help protect brain cells and reduce inflammation.

At the same time, it is advisable to avoid processed foods, excessive salt, fried items, and sugary drinks, as they can negatively impact overall health and slow down recovery.

---

## 🧘 Lifestyle Recommendations

Maintaining a healthy lifestyle is important in managing meningioma.

Regular medical checkups and MRI scans should be conducted to monitor tumor growth. Patients are encouraged to maintain a stress-free lifestyle, get adequate sleep, and engage in light physical activity.

Avoiding stress, overexertion, and unhealthy habits such as smoking or alcohol consumption is highly recommended.

---

## 📌 Preventive Measures

Although meningioma tumors are not always preventable, early detection and regular monitoring can significantly reduce complications.

Following medical advice, maintaining a healthy routine, and keeping track of symptoms can help in managing the condition effectively.

---

## 📈 Long-Term Brain Health Strategy

Long-term care includes regular follow-ups with healthcare professionals, maintaining a balanced diet, and ensuring mental and physical well-being.

Adopting healthy habits and staying informed about the condition can help improve quality of life and prevent further complications.

---

## 📋 Additional Notes

This report is generated using AI and is intended for informational purposes only. A confirmed diagnosis should always be made by a qualified medical professional.

Regular monitoring and early medical intervention are key factors in successfully managing meningioma tumors.

---

## 🧠 Conclusion

Based on the MRI analysis, the presence of a **meningioma tumor** has been detected. Although it is typically a slow-growing tumor, it requires proper medical evaluation and monitoring.

With timely medical care, lifestyle management, and regular checkups, the condition can be effectively controlled and managed over time.
"""

    # 🔴 PITUITARY
    elif disease == "pituitary":
        report = f"""
### ⚠️ Diagnosis: Pituitary Tumor Detected

Based on the analysis of the MRI scan, there is evidence suggesting the presence of a pituitary tumor. This type of tumor occurs in the pituitary gland, a small but crucial structure located at the base of the brain responsible for regulating hormones in the body.

---

## 🧠 Brain Health Condition

The scan indicates an abnormality in the region corresponding to the pituitary gland, located at the base of the brain (**{position_loc}**). This tumor may affect hormonal balance, as the pituitary gland controls several important body functions including growth, metabolism, and reproductive processes.

Although many pituitary tumors are benign (non-cancerous), they can still cause significant health issues due to hormonal imbalances or pressure on nearby structures such as the optic nerves.

---

## ⚠️ Risk Assessment

The overall condition is categorized as **Moderate Risk**.

This means that while the tumor may not be immediately life-threatening, it requires proper medical evaluation and ongoing monitoring to prevent complications related to hormone imbalance or vision problems.

---

## 🔍 Detailed Analysis

During the MRI analysis, the system detected abnormalities in the central lower region of the brain. These include slight irregularities in structure and localized density changes near the pituitary gland.

Such findings are consistent with pituitary tumors and may indicate disruption in normal gland function.

---

## 🩺 Symptoms Awareness

Pituitary tumors often affect hormone levels and can lead to a variety of symptoms, including:

- Hormonal imbalance affecting body functions
- Unexplained weight gain or loss
- Fatigue and weakness
- Vision problems, especially peripheral vision loss
- Headaches
- Irregular menstrual cycles or other endocrine issues

If any of these symptoms are observed, medical consultation with a specialist is strongly recommended.

---

## 🥗 Diet & Nutrition Recommendations

Maintaining a balanced diet is important for managing hormonal health and overall well-being.

It is recommended to consume nutrient-rich foods such as fresh fruits, vegetables, whole grains, lean proteins, and healthy fats. Adequate hydration is also essential for proper body function.

Avoid processed foods, excessive sugar, junk food, and alcohol, as these can disrupt hormonal balance and negatively impact recovery.

---

## 🧘 Lifestyle Recommendations

A stable and healthy lifestyle plays a key role in managing pituitary conditions.

Patients are advised to consult an endocrinologist for hormone evaluation and follow prescribed treatments. Regular monitoring of hormone levels and imaging tests is essential.

Maintaining proper sleep, reducing stress through relaxation techniques, and engaging in light physical activity can help improve overall health.

Avoid stress, irregular sleep patterns, and self-medication without professional guidance.

---

## 📌 Preventive Measures

While pituitary tumors may not always be preventable, early detection and proper management can significantly reduce complications.

Regular medical checkups, hormone level monitoring, and timely imaging tests are important steps in managing the condition effectively.

---

## 📈 Long-Term Brain Health Strategy

Long-term care involves maintaining hormonal balance, following medical advice, and ensuring a healthy lifestyle.

Continuous monitoring, adherence to treatment plans, and maintaining both physical and mental well-being are essential for improving quality of life.

---

## 📋 Additional Notes

This report is generated using AI and is intended for informational purposes only. A confirmed diagnosis requires evaluation by a qualified medical professional.

Early detection and proper treatment play a crucial role in managing pituitary tumors successfully.

---

## 🧠 Conclusion

Based on the MRI analysis, the presence of a **pituitary tumor** has been detected. While many such tumors are manageable, proper medical evaluation and treatment are necessary to control hormonal effects and prevent complications.

With appropriate care, monitoring, and lifestyle management, individuals can effectively manage the condition and maintain a good quality of life.
"""

    # 🔹 Extra notes
    extra_notes = [
        "Further clinical validation is recommended.",
        "Early detection improves treatment success.",
        "Regular monitoring is crucial.",
        "Consult specialists for accurate diagnosis."
    ]

    report = intro + report + "\n\n### 📌 Additional Notes\n- " + "\n- ".join(random.sample(extra_notes, 3))

    return report


# -------------------- STREAMLIT UI --------------------
st.set_page_config(page_title="BIO-X-VISION", layout="centered")

st.markdown(
    """
    <h1 style='text-align: center;'>🧠 BioVision-X</h1>
    <p style='text-align: center; font-size:16px;'>
    BioVision-X is an advanced AI-powered brain tumor detection system designed to analyze MRI scans with high accuracy.
    It utilizes deep learning techniques, including Convolutional Neural Networks (CNN), to identify and classify brain tumors
    such as glioma, meningioma, and pituitary tumors. The system enhances medical decision-making by providing visual
    explanations through Grad-CAM heatmaps, segmentation masks, and 3D brain visualization. It also generates detailed
    medical reports with risk analysis and recommendations. BioVision-X aims to assist healthcare professionals while making
    complex medical insights understandable for everyone.
    </p>
    """,
    unsafe_allow_html=True
)

uploaded_file = st.file_uploader("Upload MRI Image", type=["jpg","png","jpeg"])

if uploaded_file is not None:

    image = Image.open(uploaded_file)
    st.image(image, caption="Original MRI", use_container_width=True)

    if st.button("Analyze"):

        disease, confidence, img_array = predict_image(image)

        # Save image for 3D tumor mapping
        file_path = "temp_mri.jpg"
        image.save(file_path)

        # Visuals
        heatmap = get_gradcam_heatmap(model, img_array)
        heatmap_img = overlay_heatmap(image, heatmap)
        bbox_img = draw_bounding_box(image)
        mask = simple_segmentation(image)

        # 3D Visualization (FIXED)
        fig = visualize_fixed_brain(disease, file_path)

        # Report (FIXED)
        report = generate_report(disease, confidence)

        # Results
        st.success(f"{disease.upper()} ({confidence*100:.2f}%)")

        # 🔥 Grad-CAM
        st.markdown("<h3 style='text-align: center;'>🔥 Grad-CAM Heatmap</h3>", unsafe_allow_html=True)
        col1, col2, col3 = st.columns([1,2,1])
        with col2:
            st.image(heatmap_img)

        # 📦 Bounding Box
        st.markdown("<h3 style='text-align: center;'>📦 Bounding Box Detection</h3>", unsafe_allow_html=True)
        col1, col2, col3 = st.columns([1,2,1])
        with col2:
            st.image(bbox_img)

        # 🧠 Segmentation
        st.markdown("<h3 style='text-align: center;'>🧠 Segmentation Mask</h3>", unsafe_allow_html=True)
        col1, col2, col3 = st.columns([1,2,1])
        with col2:
            st.image(mask)

        st.subheader("🧠 3D Brain Model")
        st.plotly_chart(fig, use_container_width=True)

        st.subheader("📋 Medical Report")
        st.markdown(report)

        # Sidebar Dashboard
        st.sidebar.title("🧑‍⚕️ Doctor Dashboard")
        st.sidebar.write(f"Prediction: {disease}")
        st.sidebar.write(f"Confidence: {round(confidence*100,2)}%")

        if st.sidebar.button("Download Report"):
            with open("report.txt","w") as f:
                f.write(report)
            st.sidebar.success("Report Saved!")

Overwriting app.py


In [27]:
# 1. Kill old sessions
!pkill ngrok
# 1️⃣ Run Streamlit
!streamlit run app.py &>/content/logs.txt &

# 2️⃣ Start ngrok
from pyngrok import ngrok
ngrok.set_auth_token("2zywGxhmq42Np0vrLeN9xDCF2sA_MXPqhnVjgFqNuW5Qwx1B")

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://7dfe-34-87-120-122.ngrok-free.app" -> "http://localhost:8501"


## Testing without Streamlit ##

In [ ]:
import plotly.graph_objects as go
import numpy as np
import cv2


def obj_data_to_mesh3d(odata):
    vertices = []
    faces = []

    for line in odata.splitlines():
        vals = line.split()
        if not vals:
            continue

        if vals[0] == 'v':
            vertices.append(list(map(float, vals[1:4])))

        elif vals[0] == 'f':
            face = [int(part.split('/')[0]) - 1 for part in vals[1:]]

            if len(face) == 3:
                faces.append(face)
            else:
                for i in range(1, len(face) - 1):
                    faces.append([face[0], face[i], face[i + 1]])

    return np.array(vertices), np.array(faces)


# ----------------------------
# 🔥 STRONG tumor detection
# ----------------------------
def get_tumor_mask(image_path):
    img = cv2.imread(image_path, 0)

    if img is None:
        print("❌ Image not loaded")
        return np.zeros((256, 256), dtype=np.uint8)

    img = cv2.resize(img, (256, 256))

    # 🔥 Enhance contrast (VERY IMPORTANT)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # 🔥 Blur
    img = cv2.GaussianBlur(img, (5, 5), 0)

    # 🔥 Otsu threshold (auto)
    _, mask = cv2.threshold(img, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 🔥 Morphology (clean noise)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    print("✅ Tumor pixels detected:", np.count_nonzero(mask))

    return mask


# ----------------------------
# 🔥 Efficient 2D → 3D mapping
# ----------------------------
def map_tumor_to_3d(mask, x, y, z):

    coords = np.column_stack(np.where(mask > 0))

    if len(coords) == 0:
        print("⚠️ No tumor detected — check image")
        return np.empty((0, 3))

    # 🔥 Sample points (important)
    coords = coords[np.random.choice(len(coords),
                                     min(1500, len(coords)),
                                     replace=False)]

    idx = (coords[:, 0] * mask.shape[1] + coords[:, 1]) % len(x)

    tumor_points = np.column_stack((x[idx], y[idx], z[idx]))

    return tumor_points


def visualize_fixed_brain(disease, image_path=None):

    # ----------------------------
    # Load LEFT hemisphere
    # ----------------------------
    with open("lh.pial.obj", "r") as f:
        obj_data_lh = f.read()

    vertices_lh, faces_lh = obj_data_to_mesh3d(obj_data_lh)

    # ----------------------------
    # Mirror RIGHT hemisphere
    # ----------------------------
    vertices_rh = vertices_lh.copy()
    vertices_rh[:, 0] *= -1

    faces_rh = faces_lh + len(vertices_lh)

    # ----------------------------
    # Merge
    # ----------------------------
    vertices = np.vstack((vertices_lh, vertices_rh))
    faces = np.vstack((faces_lh, faces_rh))

    x, y, z = vertices.T
    i, j, k = faces.T

    # ----------------------------
    # Brain Mesh
    # ----------------------------
    brain = go.Mesh3d(
        x=x, y=y, z=z,
        i=i, j=j, k=k,
        color='lightgray',
        opacity=0.4,

        lighting=dict(
            ambient=0.6,
            diffuse=0.8,
            specular=0.3
        ),

        name="Brain"
    )

    fig = go.Figure(data=[brain])

    # ----------------------------
    # 🔥 REAL Tumor Highlight
    # ----------------------------
    if disease.lower() != "notumor" and image_path:

        mask = get_tumor_mask(image_path)
        tumor_coords = map_tumor_to_3d(mask, x, y, z)

        if len(tumor_coords) > 0:

            tumor = go.Scatter3d(
                x=tumor_coords[:, 0],
                y=tumor_coords[:, 1],
                z=tumor_coords[:, 2],
                mode='markers',
                marker=dict(
                    size=4,        # 🔥 bigger for visibility
                    color='red',
                    opacity=0.9
                ),
                name="Tumor Region"
            )

            fig.add_trace(tumor)

    # ----------------------------
    # Layout
    # ----------------------------
    fig.update_layout(
        title="🧠 3D Brain Tumor Visualization",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    return fig


# ----------------------------
# ✅ RUN THIS
# ----------------------------
fig = visualize_fixed_brain(
    disease="tumor",
    image_path="your_mri_image.png"  # 🔥 IMPORTANT
)

fig.show()